设置gradient_checkpointing=True可以节约显存。其主要应用了torch.utils.checkpoint.checkpoint方法。
它的原理非常简单，在对decoder_layer进行forward时不保存中间激活值从而节约显存，backward时重新计算相关值，从而通过时间换取了空间。

gradient_checkpointing和use_cache不能同时设置为True，前者是为了节约显存时间换空间的，后者是为了节约时间空间换时间。

参考资料：

https://zhuanlan.zhihu.com/p/448395808

主要看这个  https://zhuanlan.zhihu.com/p/615122110

# 工作机制

正常训练时，模型会保存所有层的中间激活值用于反向传播

使用梯度检查点时，只保存特定检查点的激活值

需要其他激活值时，通过重新计算获得

优点：
1 显著减少显存使用（可减少高达80%的显存占用）
2 允许训练更大的模型或使用更大的batch size

缺点：
1 需要重新计算部分前向传播，增加了计算时间（约30%左右）
2 会消耗更多CPU内存

# example

使用 PyTorch 实现模型或部分模型的检查点技术非常简单。可以将需要应用检查点技术的模块（nn.module）封装在 torch.utils.checkpoint.checkpoint() 函数中，然后将其用作前向传递的函数即可。例如，下面的代码将应用检查点技术的模块 module 作为前向传递函数进行检查点处理：

假设我们有一个包含 4 个线性层的神经网络，批量大小为 2 的情况下进行训练。我们可以将网络中的其中一层线性层来应用检查点技术，以减少内存的使用

In [1]:
import torch
import torch.nn as nn
import torch.utils.checkpoint as cp

# define the neural network
class MyNet(nn.Module):
    def __init__(self):
        super(MyNet, self).__init__()
        self.fc1 = nn.Linear(784, 1024)
        self.fc2 = nn.Linear(1024, 1024)
        self.fc3 = nn.Linear(1024, 512)
        self.fc4 = nn.Linear(512, 10)

    def forward(self, x):
        # apply the first 2 layers
        x = nn.functional.relu(self.fc1(x))
        x = nn.functional.relu(self.fc2(x))
        # apply the checkpointed layers
        x = cp.checkpoint(self._checkpointed_forward, x)
        # apply the last 1 layers
        x = self.fc4(x)
        return x

    def _checkpointed_forward(self, x):
        x = nn.functional.relu(self.fc3(x))
        return x

# create a random input tensor
x = torch.randn(2, 784)

# create the neural network
model = MyNet()

# compute the output
output = model(x)

# compute the gradients
output.sum().backward()

# print the gradients of the first layer
print(model.fc1.weight.grad)

tensor([[-0.0100,  0.0084,  0.0056,  ..., -0.0063, -0.0012, -0.0009],
        [-0.0064,  0.0240,  0.0081,  ..., -0.0262, -0.0007, -0.0178],
        [-0.0039, -0.0050,  0.0002,  ...,  0.0073, -0.0005,  0.0073],
        ...,
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0046, -0.0232, -0.0072,  ...,  0.0259,  0.0005,  0.0182],
        [ 0.0112, -0.0152, -0.0076,  ...,  0.0140,  0.0014,  0.0063]])


e:\miniconda3\envs\llm\Lib\site-packages\torch\utils\checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


一些例子可以看llama源码

# 总结

简单来说，如果我们想要训练大模型，那么就可以牺牲一些训练时间，来换取模型可以放进显存，此时只需要简单的调用torch.utils.checkpoint.checkpoint 函数来包装想要节省显存的模块部分。此外并不需要做额外的操作，在实际使用中，可以再训练脚本中设置一个configs args 例如：

parser.add_argument('--use_checkpoint', action='store_true')
然后将代码包装成：

In [ ]:
self.use_checkpoint = args.use_checkpoint
...

if self.use_checkpoint:
    output = checkpoint.checkpoint(layer, inputs)
else:
    output = layer(inputs)